# OS & Shell Tutor: Training Notebook (Qwen3 + Unsloth + LoRA)

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU`
2. In Google Drive create a folder `os_tutor` and upload your dataset (`synthetic.json`) into it.
3. Run the cells **top to bottom** (Shift+Enter). Edit only the **Config** cell.

Your data must be in chat format:
```json
{"messages": [
  {"role": "system", "content": "You are an OS and shell tutor."},
  {"role": "user", "content": "What is a deadlock?"},
  {"role": "assistant", "content": "A deadlock is..."}
]}
```

## 1. Check the GPU

In [1]:
!nvidia-smi

Tue Oct 13 17:43:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install libraries
If Colab asks you to restart the session afterwards, do it, then continue from the next cell.

In [2]:
!pip install -q unsloth

## 3. Mount Google Drive
So your data and trained model are saved permanently (Colab's local disk is wiped when the session ends).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Config (edit this cell only)

In [4]:
BASE_MODEL = "unsloth/Qwen3-1.7B"       # try "unsloth/Qwen3-1.7B" if you run out of memory
MAX_SEQ_LEN  = 2048

# Add more files here later, e.g. ["synthetic.json", "manual_verified.json"]
DATA_FILES   = ["/content/drive/MyDrive/os_tutor/synthetic.json"]

# Change the version name for each training run: v1, v2, ...
SAVE_DIR     = "/content/drive/MyDrive/os_tutor/os_tutor_v1"

EPOCHS       = 2        # more than 3 on small synthetic data usually overfits
LEARNING_RATE = 2e-4

## 5. Load, clean and split the dataset
- **Deduplicate** on the user question so the model doesn't memorize repeats
- **Shuffle** so it doesn't learn topic order
- **5% validation split**: data the model never trains on. If validation loss rises while training loss falls, you're overfitting.

In [5]:
import json, random
from datasets import Dataset

def load_file(path):
    with open(path) as f:
        text = f.read().strip()
    if text.startswith("["):                 # normal JSON list
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]   # JSONL

data = []
for p in DATA_FILES:
    items = load_file(p)
    print(p, "->", len(items), "examples")
    data += items

print("\nFirst example:\n", json.dumps(data[0], indent=2)[:800])

/content/drive/MyDrive/os_tutor/synthetic.json -> 52 examples

First example:
 {
  "messages": [
    {
      "role": "system",
      "content": "You are an OS and shell tutor. Explain step by step and be accurate."
    },
    {
      "role": "user",
      "content": "What is the difference between a process and a thread?"
    },
    {
      "role": "assistant",
      "content": "A **process** is a program in execution with its own address space (code, data, heap, stack) and OS resources such as open files. A **thread** is a unit of execution *inside* a process.\n\n- **Sharing:** threads of one process share code, data, heap and open files, but each thread has its own stack, registers and program counter.\n- **Cost:** creating a thread and switching between threads is cheaper than for processes.\n- **Isolation:** processes are isolated from each other. A bug in one th


In [6]:
def user_question(ex):
    return next(m["content"] for m in ex["messages"] if m["role"] == "user")

seen, clean = set(), []
for ex in data:
    key = user_question(ex).strip().lower()
    if key not in seen:
        seen.add(key)
        clean.append(ex)

# keep only the "messages" field so the dataset is uniform
clean = [{"messages": ex["messages"]} for ex in clean]

random.seed(42)
random.shuffle(clean)

ds = Dataset.from_list(clean).train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = ds["train"], ds["test"]
print(f"{len(data)} loaded -> {len(clean)} after dedup")
print(f"{len(train_ds)} train / {len(val_ds)} validation")

52 loaded -> 52 after dedup
49 train / 3 validation


**Using a different format?** If your file has `instruction` / `output` instead of `messages`, run this converter *before* the dedup cell (uncomment it):

In [7]:
# data = [{"messages": [
#     {"role": "system", "content": "You are an OS and shell tutor."},
#     {"role": "user", "content": ex["instruction"]},
#     {"role": "assistant", "content": ex["output"]},
# ]} for ex in data]

## 6. Load the base model in 4-bit
- `load_in_4bit=True` compresses weights so the model fits the T4's 16 GB (this is QLoRA).
- `max_seq_length` is the longest example (in tokens) the model handles.

In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

## 7. Attach LoRA adapters
LoRA freezes the base model and trains small add-on matrices instead of all the weights.
- `r`: adapter size (higher = more capacity, more memory)
- `lora_alpha`: scaling factor, usually equal to `r`
- `target_modules`: which layers get adapters

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.10.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 8. Apply the chat template
Converts message lists into the exact text format Qwen3 expects (with tokens like `<|im_start|>`).
**Use the same template at inference time**; a mismatch is a classic cause of a "bad" fine-tune.

In [10]:
def format_example(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_ds = train_ds.map(format_example)
val_ds = val_ds.map(format_example)
print(train_ds[0]["text"])

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

<|im_start|>system
You are an OS and shell tutor. Explain step by step and be accurate.<|im_end|>
<|im_start|>user
What is the difference between a mutex and a semaphore?<|im_end|>
<|im_start|>assistant
<think>

</think>

Both are synchronization tools, but they are used differently.

- **Mutex:** a lock with *ownership*. Only the thread that locked it may unlock it. It protects a critical section so only one thread is inside at a time.
- **Semaphore:** an integer counter with two atomic operations, `wait()` (decrement, block if the value would go below 0) and `signal()` (increment). Any thread may call `signal()`.
  - A **binary semaphore** (0 or 1) can act like a lock.
  - A **counting semaphore** manages N identical resources, for example 5 database connections.

Rule of thumb: use a mutex for mutual exclusion, and a semaphore for counting resources or for signalling between threads (for example, a producer telling a consumer that an item is ready).<|im_end|>



## 9. Set up the trainer
- Batch size 2 x gradient accumulation 4 = effective batch of 8
- `train_on_responses_only` makes the model learn from the **assistant answers only**, not the questions
- Validation loss is printed every 50 steps

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,      # on older trl versions use: tokenizer=tokenizer
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/49 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/49 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

## 10. Train
Healthy training: both training and validation loss go **down**. If validation loss starts rising while training loss keeps falling, use fewer epochs.

In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 2 | Total steps = 14
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
14,2.233071,5.008521


Filter:   0%|          | 0/3 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-14/tokenizer_config.json.


TrainOutput(global_step=14, training_loss=2.064394337790353, metrics={'train_runtime': 36.2624, 'train_samples_per_second': 2.703, 'train_steps_per_second': 0.386, 'total_flos': 175518764863488.0, 'train_loss': 2.064394337790353, 'epoch': 2.0})

## 11. Quick test

In [13]:
FastLanguageModel.for_inference(model)   # switch to fast generation mode

def ask(question, system="You are an OS and shell tutor.", max_new_tokens=512):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.8, top_k=20, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("Explain the difference between a process and a thread."))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### **Process vs. Thread**

**Process** is a program in execution, and it contains **threads**.

**Thread** is a **unit of execution** within a process.

**Process** is a **program in execution**.

**Thread** is a **unit of execution** within a **process**.

### **Process**
- **Process** is a **program in execution**.
- It has a **program counter**, **stack**, **program code**, **heap**, **environment**.
- It has **process ID (PID)**.
- It is **not** a unit of execution.
- It has **process control block (PCB)**.

**Thread**
- A **unit of execution** within a **process**.
- It has **program counter**, **stack**, **program code**, **heap**, **environment**.
- It has **thread ID**.
- It is **a unit of execution**.
- It has **thread control block (TCB)**.

### **Process**
- A **process** is a **program in execution**.
- It is **not** a unit of execution.
- It is **not** a unit of control.
- It has **process control block (PCB)**.
- It is **not** a unit of execution.
- It has **process ID (

## 12. Save the LoRA adapter to Drive
The adapter is small (usually under 200 MB). **This is what the chatbot notebook loads.**

In [18]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

Saved to /content/drive/MyDrive/os_tutor/os_tutor_v1


## 13. (Optional) Export a merged model or GGUF
Only needed if you want to use the model outside Unsloth (Transformers, vLLM, LM Studio, llama.cpp). GGUF export builds llama.cpp in the background and can take several minutes.

In [15]:
# Merged 16-bit model (adapter baked into base weights)
# model.save_pretrained_merged(SAVE_DIR + "_merged", tokenizer, save_method="merged_16bit")

# GGUF for LM Studio / llama.cpp
# model.save_pretrained_gguf(SAVE_DIR + "_gguf", tokenizer, quantization_method="q4_k_m")

In [19]:
!pip install -q unsloth
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "/content/drive/MyDrive/os_tutor/os_tutor_v1",   # your saved adapter
    max_seq_length=2048,
    load_in_4bit=True,
)

model.save_pretrained_gguf(
    "/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 266.64it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:38<00:00, 98.74s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf_gguf/Qwen3-1.7B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/os_tutor/os_tutor_v1_

{'save_directory': '/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf',
 'gguf_directory': '/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf_gguf',
 'gguf_files': ['/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf_gguf/Qwen3-1.7B.Q4_K_M.gguf'],
 'modelfile_location': '/content/drive/MyDrive/os_tutor/os_tutor_v1_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## Next steps
- **Add more data:** put new files in `DATA_FILES` (Config cell), change `SAVE_DIR` to `..._v2`, and use **Runtime -> Restart session and run all**. Retraining from the base model on old + new data is the safest approach.
- Open the **`os_tutor_rag_chatbot`** notebook to add RAG and the chat interface.